# Macro Feature Neural Models

Trains the LSTM and graph neural models on the separate `stock_features_plus_regime_v1` dataset. HAR baselines remain unchanged.

In [1]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch

import config
from src.macro_dataset import load_feature_tensor, MACRO_FEATURE_VERSION
from src.models import LSTMModel, GNNModel, GNNModelV2
from src.train import (
    set_seeds, train_lstm, predict_lstm_split, train_gnn, predict_gnn_split,
)
from src.graphs import load_corr_graphs
from src.experiment_registry import register_experiment

results_dir = Path(config.DATA_RESULTS_DIR)
ckpt_dir = Path(config.CHECKPOINTS_DIR)
features_dir = Path(config.DATA_FEATURES_DIR)
graphs_dir = Path(config.DATA_GRAPHS_DIR)
results_dir.mkdir(parents=True, exist_ok=True)
ckpt_dir.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


In [2]:
# Load macro feature tensor and aligned target/splits
target = pd.read_parquet(features_dir / 'target.parquet')
target.index = pd.to_datetime(target.index)
splits = pd.read_parquet(features_dir / 'splits.parquet')
splits['week'] = pd.to_datetime(splits['week'])

features_3d, feature_names, tickers, week_index = load_feature_tensor(
    feature_path=features_dir / 'features_macro.parquet',
    meta_path=features_dir / 'features_macro_meta.json',
    target_index=target.index,
)
target = target.loc[week_index, tickers]
target_arr = target.to_numpy(dtype=float)

train_pos = np.flatnonzero(week_index.isin(splits.loc[splits['split'].eq('train'), 'week']))
val_pos = np.flatnonzero(week_index.isin(splits.loc[splits['split'].eq('val'), 'week']))
test_pos = np.flatnonzero(week_index.isin(splits.loc[splits['split'].eq('test'), 'week']))

n_weeks, n_stocks, n_feats = features_3d.shape
print('Macro tensor:', features_3d.shape)
print('Feature version:', MACRO_FEATURE_VERSION)
print('Train/val/test positions:', len(train_pos), len(val_pos), len(test_pos))
print('Last five features:', feature_names[-5:])

Macro tensor: (572, 465, 19)
Feature version: stock_features_plus_regime_v1
Train/val/test positions: 417 52 103
Last five features: ['spy_return_1m', 'treasury_10y_2y_spread', 'ig_credit_spread', 'avg_pairwise_stock_correlation', 'correlation_graph_density']


In [3]:
# Graph loaders
zero_edges = torch.zeros((2, 0), dtype=torch.long)

corr_graphs = {}
for split_name in ('train', 'val', 'test'):
    corr_graphs.update(load_corr_graphs(config.CORR_THRESHOLD, split_name))

def corr_edge_index(week):
    return corr_graphs.get(pd.Timestamp(week), zero_edges)

sector_edges = pd.read_parquet(graphs_dir / 'sector_edges_by_year.parquet')
sector_graphs = {}
for year, group in sector_edges.groupby('year'):
    sector_graphs[int(year)] = torch.tensor(group[['src', 'dst']].to_numpy(dtype=np.int64).T, dtype=torch.long)

def sector_edge_index(week):
    return sector_graphs[int(pd.Timestamp(week).year)]

granger_edges = pd.read_parquet(graphs_dir / 'granger_edges.parquet')
granger_graph = torch.tensor(granger_edges[['src', 'dst']].to_numpy(dtype=np.int64).T, dtype=torch.long)

def granger_edge_index(week):
    return granger_graph

print('Correlation graph weeks:', len(corr_graphs))
print('Sector graph years:', sorted(sector_graphs))
print('Granger edges:', granger_graph.shape[1])

Correlation graph weeks: 572
Sector graph years: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Granger edges: 13886


In [4]:
# LSTM + Macro
set_seeds()
lstm = LSTMModel(input_size=n_feats).to(device)
lstm_ckpt = ckpt_dir / 'lstm_macro_best.pt'
lstm_loss_path = results_dir / 'lstm_macro_val_loss.json'

if lstm_ckpt.exists() and lstm_loss_path.exists():
    print('lstm_macro_best.pt found; loading saved checkpoint.')
    lstm.load_state_dict(torch.load(lstm_ckpt, map_location=device, weights_only=True))
    lstm_val_loss = json.load(open(lstm_loss_path))['val_loss']
else:
    lstm, lstm_val_loss = train_lstm(
        lstm, features_3d, target_arr, train_pos, val_pos, device, checkpoint_prefix='lstm_macro'
    )

lstm_val_preds = predict_lstm_split(lstm, features_3d, week_index, splits, tickers, device, split_name='val')
lstm_test_preds = predict_lstm_split(lstm, features_3d, week_index, splits, tickers, device, split_name='test')
lstm_val_preds.to_parquet(results_dir / 'lstm_macro_val_preds.parquet')
lstm_test_preds.to_parquet(results_dir / 'test_preds_lstm_macro.parquet')
print('LSTM + Macro best val MSE:', min(lstm_val_loss))
print('Saved LSTM macro predictions:', lstm_val_preds.shape, lstm_test_preds.shape)

C:\Users\Rylan Wade\Desktop\FinancialAnalytics\Final_Project\.venv\lib\site-packages\torch\optim\lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch   1  train=0.036575  val=0.021331


Epoch   2  train=0.031036  val=0.019973


Epoch   3  train=0.028356  val=0.019875


Epoch   4  train=0.027483  val=0.020082


...

Epoch  25  train=0.021773  val=0.020241


Epoch  26  train=0.021583  val=0.020154


Epoch  27  train=0.021520  val=0.020232
Early stop at epoch 27 (no improvement for 10 epochs)


C:\Users\Rylan Wade\Desktop\FinancialAnalytics\Final_Project\src\train.py:240: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_path, map_

LSTM + Macro best val MSE: 0.01959391203350746
Saved LSTM macro predictions: (52, 465) (103, 465)


In [5]:
# GNN-Correlation + Macro
set_seeds()
gnn_corr = GNNModelV2(
    in_channels=n_feats,
    hidden_dim=config.HIDDEN_DIM,
    dropout=config.DROPOUT,
    num_layers=config.GNN_NUM_LAYERS,
    batch_norm=False,
).to(device)

if (ckpt_dir / 'gnn_corr_macro_best.pt').exists() and (results_dir / 'gnn_corr_macro_val_loss.json').exists():
    print('gnn_corr_macro_best.pt found; loading saved checkpoint.')
    gnn_corr.load_state_dict(torch.load(ckpt_dir / 'gnn_corr_macro_best.pt', map_location=device, weights_only=True))
    corr_val_loss = json.load(open(results_dir / 'gnn_corr_macro_val_loss.json'))['val_loss']
else:
    gnn_corr, corr_val_loss = train_gnn(
        gnn_corr, features_3d, target_arr, week_index, corr_edge_index, splits,
        graph_type='corr_macro', device=device, max_epochs=config.GNN_MAX_EPOCHS,
        lr=config.GNN_LEARNING_RATE, patience=config.EARLY_STOP_PATIENCE, save_periodic=False,
    )

gnn_corr_val = predict_gnn_split(gnn_corr, features_3d, week_index, corr_edge_index, splits, tickers, device, split_name='val')
gnn_corr_test = predict_gnn_split(gnn_corr, features_3d, week_index, corr_edge_index, splits, tickers, device, split_name='test')
gnn_corr_val.to_parquet(results_dir / 'gnn_corr_macro_val_preds.parquet')
gnn_corr_test.to_parquet(results_dir / 'test_preds_gnn_corr_macro.parquet')
print('GNN-Correlation + Macro best val MSE:', min(corr_val_loss))

C:\Users\Rylan Wade\Desktop\FinancialAnalytics\Final_Project\src\models.py:350: UserWarning: GNNModelV2.forward: input features contain NaN values. Imputing with 0 (z-score mean). Expected during warm-up weeks.
  warnings.warn(


Epoch   1  train=0.035880  val=0.020646


Epoch   2  train=0.035263  val=0.020532


Epoch   3  train=0.031024  val=0.020556


Epoch   4  train=0.030420  val=0.020528


...

Epoch  14  train=0.029527  val=0.021028


Epoch  15  train=0.027326  val=0.020615


Epoch  16  train=0.029269  val=0.020642


Epoch  17  train=0.026267  val=0.020721
Early stop at epoch 17 (no improvement for 10 epochs)


GNN-Correlation + Macro best val MSE: 0.020163739673220195


In [6]:
# GNN-Sector + Macro
set_seeds()
gnn_sector = GNNModel(in_channels=n_feats).to(device)

if (ckpt_dir / 'gnn_sector_macro_best.pt').exists() and (results_dir / 'gnn_sector_macro_val_loss.json').exists():
    print('gnn_sector_macro_best.pt found; loading saved checkpoint.')
    gnn_sector.load_state_dict(torch.load(ckpt_dir / 'gnn_sector_macro_best.pt', map_location=device, weights_only=True))
    sector_val_loss = json.load(open(results_dir / 'gnn_sector_macro_val_loss.json'))['val_loss']
else:
    gnn_sector, sector_val_loss = train_gnn(
        gnn_sector, features_3d, target_arr, week_index, sector_edge_index, splits,
        graph_type='sector_macro', device=device, max_epochs=config.GNN_MAX_EPOCHS,
        lr=config.LEARNING_RATE, patience=config.EARLY_STOP_PATIENCE, save_periodic=False,
    )

gnn_sector_val = predict_gnn_split(gnn_sector, features_3d, week_index, sector_edge_index, splits, tickers, device, split_name='val')
gnn_sector_test = predict_gnn_split(gnn_sector, features_3d, week_index, sector_edge_index, splits, tickers, device, split_name='test')
gnn_sector_val.to_parquet(results_dir / 'gnn_sector_macro_val_preds.parquet')
gnn_sector_test.to_parquet(results_dir / 'test_preds_gnn_sector_macro.parquet')
print('GNN-Sector + Macro best val MSE:', min(sector_val_loss))

C:\Users\Rylan Wade\Desktop\FinancialAnalytics\Final_Project\src\models.py:289: UserWarning: GNNModel.forward: input features contain NaN values. Imputing with 0 (z-score mean). Expected during warm-up weeks.
  warnings.warn(


Epoch   1  train=0.043205  val=0.022650


Epoch   2  train=0.032117  val=0.021015


Epoch   3  train=0.030696  val=0.020854


Epoch   4  train=0.028927  val=0.020043


...

Epoch  11  train=0.027858  val=0.020668


Epoch  12  train=0.025059  val=0.020560


Epoch  13  train=0.024148  val=0.020270


Epoch  14  train=0.023694  val=0.020130
Early stop at epoch 14 (no improvement for 10 epochs)


GNN-Sector + Macro best val MSE: 0.02004285488062753


In [7]:
# GNN-Granger + Macro
set_seeds()
gnn_granger = GNNModel(in_channels=n_feats).to(device)

if (ckpt_dir / 'gnn_granger_macro_best.pt').exists() and (results_dir / 'gnn_granger_macro_val_loss.json').exists():
    print('gnn_granger_macro_best.pt found; loading saved checkpoint.')
    gnn_granger.load_state_dict(torch.load(ckpt_dir / 'gnn_granger_macro_best.pt', map_location=device, weights_only=True))
    granger_val_loss = json.load(open(results_dir / 'gnn_granger_macro_val_loss.json'))['val_loss']
else:
    gnn_granger, granger_val_loss = train_gnn(
        gnn_granger, features_3d, target_arr, week_index, granger_edge_index, splits,
        graph_type='granger_macro', device=device, max_epochs=config.GNN_MAX_EPOCHS,
        lr=config.LEARNING_RATE, patience=config.EARLY_STOP_PATIENCE, save_periodic=False,
    )

gnn_granger_val = predict_gnn_split(gnn_granger, features_3d, week_index, granger_edge_index, splits, tickers, device, split_name='val')
gnn_granger_test = predict_gnn_split(gnn_granger, features_3d, week_index, granger_edge_index, splits, tickers, device, split_name='test')
gnn_granger_val.to_parquet(results_dir / 'gnn_granger_macro_val_preds.parquet')
gnn_granger_test.to_parquet(results_dir / 'test_preds_gnn_granger_macro.parquet')
print('GNN-Granger + Macro best val MSE:', min(granger_val_loss))

Epoch   1  train=0.043152  val=0.023135


Epoch   2  train=0.032225  val=0.021003


Epoch   3  train=0.030708  val=0.021728


Epoch   4  train=0.029473  val=0.020403


Epoch   5  train=0.031959  val=0.020239


...

Epoch  34  train=0.030978  val=0.020224


Epoch  35  train=0.027184  val=0.020180


Epoch  36  train=0.025846  val=0.020136


Epoch  37  train=0.024783  val=0.020235
Early stop at epoch 37 (no improvement for 10 epochs)


GNN-Granger + Macro best val MSE: 0.01987525270893597


In [8]:
# GNN-Ensemble + Macro
gnn_ensemble_val = (gnn_corr_val + gnn_sector_val + gnn_granger_val) / 3.0
gnn_ensemble_test = (gnn_corr_test + gnn_sector_test + gnn_granger_test) / 3.0
gnn_ensemble_val.to_parquet(results_dir / 'gnn_ensemble_macro_val_preds.parquet')
gnn_ensemble_test.to_parquet(results_dir / 'test_preds_gnn_ensemble_macro.parquet')
print('Saved GNN-Ensemble + Macro predictions:', gnn_ensemble_val.shape, gnn_ensemble_test.shape)

Saved GNN-Ensemble + Macro predictions: (52, 465) (103, 465)


In [9]:
# Register macro experiments
def _json(value):
    return json.dumps(value, sort_keys=True, separators=(',', ':'))

base_hparams = {
    'random_seed': config.RANDOM_SEED,
    'data_start': config.DATA_START,
    'data_end': config.DATA_END,
    'feature_artifact': 'data/features/features_macro.parquet',
    'feature_meta': 'data/features/features_macro_meta.json',
}
common = {
    'feature_version': MACRO_FEATURE_VERSION,
    'train_split': f'{config.DATA_START} through {config.TRAIN_END}',
    'val_split': f'{config.TRAIN_END} exclusive through {config.VAL_END}',
    'test_split': f'{config.VAL_END} exclusive through {config.TEST_END}',
    'test_metrics_path': '',
    'portfolio_metrics_path': '',
}
rows = [
    {**common, 'experiment_id': 'macro_lstm', 'model_name': 'LSTM + Macro', 'model_family': 'LSTM', 'graph_type': 'none', 'loss_type': 'mse', 'graph_version': 'none', 'checkpoint_path': 'data/results/checkpoints/lstm_macro_best.pt', 'hyperparameters': _json({**base_hparams, 'hidden_dim': config.LSTM_HIDDEN_DIM, 'seq_len': config.LSTM_SEQ_LEN, 'learning_rate': config.LEARNING_RATE}), 'validation_metrics_path': 'data/results/lstm_macro_val_loss.json', 'prediction_path': 'data/results/test_preds_lstm_macro.parquet', 'validation_prediction_path': 'data/results/lstm_macro_val_preds.parquet', 'notes': 'Macro/regime feature LSTM trained on stock_features_plus_regime_v1.'},
    {**common, 'experiment_id': 'macro_gnn_correlation', 'model_name': 'GNN-Correlation + Macro', 'model_family': 'GNN', 'graph_type': 'correlation', 'loss_type': 'mse', 'graph_version': f'correlation_threshold_{config.CORR_THRESHOLD}_lookback_{config.CORR_LOOKBACK_DAYS}', 'checkpoint_path': 'data/results/checkpoints/gnn_corr_macro_best.pt', 'hyperparameters': _json({**base_hparams, 'architecture': 'GNNModelV2', 'hidden_dim': config.HIDDEN_DIM, 'num_layers': config.GNN_NUM_LAYERS, 'dropout': config.DROPOUT, 'learning_rate': config.GNN_LEARNING_RATE}), 'validation_metrics_path': 'data/results/gnn_corr_macro_val_loss.json', 'prediction_path': 'data/results/test_preds_gnn_corr_macro.parquet', 'validation_prediction_path': 'data/results/gnn_corr_macro_val_preds.parquet', 'notes': 'Macro/regime feature correlation GNN.'},
    {**common, 'experiment_id': 'macro_gnn_sector', 'model_name': 'GNN-Sector + Macro', 'model_family': 'GNN', 'graph_type': 'sector', 'loss_type': 'mse', 'graph_version': 'sector_canonical_gics_labels_v1', 'checkpoint_path': 'data/results/checkpoints/gnn_sector_macro_best.pt', 'hyperparameters': _json({**base_hparams, 'architecture': 'GNNModel', 'hidden_dim': config.HIDDEN_DIM, 'dropout': config.DROPOUT, 'learning_rate': config.LEARNING_RATE}), 'validation_metrics_path': 'data/results/gnn_sector_macro_val_loss.json', 'prediction_path': 'data/results/test_preds_gnn_sector_macro.parquet', 'validation_prediction_path': 'data/results/gnn_sector_macro_val_preds.parquet', 'notes': 'Macro/regime feature sector GNN.'},
    {**common, 'experiment_id': 'macro_gnn_granger', 'model_name': 'GNN-Granger + Macro', 'model_family': 'GNN', 'graph_type': 'granger', 'loss_type': 'mse', 'graph_version': f'granger_lag_{config.GRANGER_LAG}_{config.GRANGER_CORRECTION}', 'checkpoint_path': 'data/results/checkpoints/gnn_granger_macro_best.pt', 'hyperparameters': _json({**base_hparams, 'architecture': 'GNNModel', 'hidden_dim': config.HIDDEN_DIM, 'dropout': config.DROPOUT, 'learning_rate': config.LEARNING_RATE}), 'validation_metrics_path': 'data/results/gnn_granger_macro_val_loss.json', 'prediction_path': 'data/results/test_preds_gnn_granger_macro.parquet', 'validation_prediction_path': 'data/results/gnn_granger_macro_val_preds.parquet', 'notes': 'Macro/regime feature Granger GNN.'},
    {**common, 'experiment_id': 'macro_gnn_ensemble', 'model_name': 'GNN-Ensemble + Macro', 'model_family': 'GNN ensemble', 'graph_type': 'ensemble', 'loss_type': 'mse', 'graph_version': 'correlation+sector+granger_v1', 'checkpoint_path': '', 'hyperparameters': _json({**base_hparams, 'constituents': ['macro_gnn_correlation', 'macro_gnn_sector', 'macro_gnn_granger'], 'combination': 'average_predictions'}), 'validation_metrics_path': '', 'prediction_path': 'data/results/test_preds_gnn_ensemble_macro.parquet', 'validation_prediction_path': 'data/results/gnn_ensemble_macro_val_preds.parquet', 'notes': 'Average prediction ensemble of macro GNN variants.'},
]
for row in rows:
    register_experiment(row, overwrite=True)
print('Registered macro experiments:', [row['experiment_id'] for row in rows])

Registered macro experiments: ['macro_lstm', 'macro_gnn_correlation', 'macro_gnn_sector', 'macro_gnn_granger', 'macro_gnn_ensemble']


In [10]:
# Training summary
summary = pd.DataFrame([
    {'model': 'LSTM + Macro', 'best_val_mse': min(lstm_val_loss), 'checkpoint': 'data/results/checkpoints/lstm_macro_best.pt'},
    {'model': 'GNN-Correlation + Macro', 'best_val_mse': min(corr_val_loss), 'checkpoint': 'data/results/checkpoints/gnn_corr_macro_best.pt'},
    {'model': 'GNN-Sector + Macro', 'best_val_mse': min(sector_val_loss), 'checkpoint': 'data/results/checkpoints/gnn_sector_macro_best.pt'},
    {'model': 'GNN-Granger + Macro', 'best_val_mse': min(granger_val_loss), 'checkpoint': 'data/results/checkpoints/gnn_granger_macro_best.pt'},
    {'model': 'GNN-Ensemble + Macro', 'best_val_mse': np.nan, 'checkpoint': ''},
])
display(summary)
summary.to_csv(results_dir / 'macro_model_training_summary.csv', index=False)
print('Saved data/results/macro_model_training_summary.csv')

,model,best_val_mse,checkpoint
0,LSTM + Macro,0.019594,data/results/checkpoints/lstm_macro_best.pt
1,GNN-Correlation + Macro,0.020164,data/results/checkpoints/gnn_corr_macro_best.pt
2,GNN-Sector + Macro,0.020043,data/results/checkpoints/gnn_sector_macro_best.pt
3,GNN-Granger + Macro,0.019875,data/results/checkpoints/gnn_granger_macro_bes...
4,GNN-Ensemble + Macro,NaN,


Saved data/results/macro_model_training_summary.csv
